# ColabNAS code

In [ ]:
import tensorflow as tf
import numpy as np
import subprocess
import datetime
import glob
import re
import os

class ColabNAS :
    architecture_name = 'resulting_architecture'
    def __init__(self, max_RAM, max_Flash, max_MACC, path_to_training_set, val_split, cache=False, input_shape=(50,50,3), save_path='./') :
        self.learning_rate = 1e-3
        self.batch_size = 128
        self.epochs = 100 #minimum 2

        self.max_MACC = max_MACC
        self.max_Flash = max_Flash
        self.max_RAM = max_RAM
        self.path_to_training_set = path_to_training_set
        self.num_classes = len(next(os.walk(path_to_training_set))[1])
        self.val_split = val_split
        self.cache = cache
        self.input_shape = input_shape
        self.save_path = save_path

        self.path_to_trained_models = f"{self.save_path}/trained_models"
        os.makedirs(self.path_to_trained_models)

        self.load_training_set()

    # k number of kernels of the first convolutional layer
    # c number of cells added upon the first convolutional layer
    # pre-processing pipeline not included in MACC computation
    def Model(self, k, c) :
        kernel_size = (3,3)
        pool_size = (2,2)
        pool_strides = (2,2)

        number_of_cells_limited = False
        number_of_mac = 0

        inputs = tf.keras.Input(shape=self.input_shape)

        #preprocessing pipeline
        x = tf.keras.layers.RandomFlip('horizontal')(inputs)
        x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
        x = tf.keras.layers.Rescaling(1./255)(x)
        x = tf.keras.layers.BatchNormalization()(x)

        #convolutional base
        n = k
        multiplier = 2

        #first convolutional layer
        c_in = self.input_shape[2]
        x = tf.keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
        number_of_mac = number_of_mac + (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        #adding cells
        for i in range(1, c + 1) :
            if x.shape[1] <= 1 or x.shape[2] <= 1 :
                number_of_cells_limited = True
                break;
            n = np.ceil(n * multiplier)
            multiplier = multiplier - 2**-i
            x = tf.keras.layers.MaxPooling2D(pool_size=pool_size, strides=pool_strides, padding='valid')(x)
            c_in = x.shape[3]
            x = tf.keras.layers.Conv2D(n, kernel_size, activation='relu', padding='same')(x)
            number_of_mac = number_of_mac + (c_in * kernel_size[0] * kernel_size[1] * x.shape[1] * x.shape[2] * x.shape[3])

        #classifier
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        input_shape = x.shape[1]
        x = tf.keras.layers.Dense(n, activation='relu')(x)
        number_of_mac = number_of_mac + (input_shape * x.shape[1])
        outputs = tf.keras.layers.Dense(self.num_classes, activation='softmax')(x)
        number_of_mac = number_of_mac + (x.shape[1] * outputs.shape[1])

        model = tf.keras.Model(inputs=inputs, outputs=outputs)

        opt = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        model.compile(optimizer=opt,
                loss='categorical_crossentropy',
                metrics=['accuracy'])

        model.summary()

        return model, number_of_mac, number_of_cells_limited

    def load_training_set(self):
        if 3 == self.input_shape[2] :
            color_mode = 'rgb'
        elif 1 == self.input_shape[2] :
            color_mode = 'grayscale'

        train_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=11,
            validation_split=self.val_split,
            subset='training'
        )

        validation_ds = tf.keras.utils.image_dataset_from_directory(
            directory= self.path_to_training_set,
            labels='inferred',
            label_mode='categorical',
            color_mode=color_mode,
            batch_size=self.batch_size,
            image_size=self.input_shape[0:2],
            shuffle=True,
            seed=11,
            validation_split=self.val_split,
            subset='validation'
        )

        if self.cache :
            self.train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
            self.validation_ds = validation_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
        else :
            self.train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
            self.validation_ds = validation_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

    def quantize_model_uint8(self) :
        def representative_dataset():
            for data in self.train_ds.rebatch(1).take(150) :
                yield [tf.dtypes.cast(data[0], tf.float32)]

        model = tf.keras.models.load_model(f"{self.path_to_trained_models}/{self.model_name}.h5")
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
        tflite_quant_model = converter.convert()

        with open(f"{self.path_to_trained_models}/{self.model_name}.tflite", 'wb') as f:
            f.write(tflite_quant_model)

        os.remove(f"{self.path_to_trained_models}/{self.model_name}.h5")

    def evaluate_flash_and_peak_RAM_occupancy(self) :
        #quantize model to evaluate its peak RAM occupancy and its Flash occupancy
        self.quantize_model_uint8()

        #evaluate its peak RAM occupancy and its Flash occupancy using STMicroelectronics' X-CUBE-AI
        proc = subprocess.Popen(["./stm32tflm", f"{self.path_to_trained_models}/{self.model_name}.tflite"], stdout=subprocess.PIPE)
        try:
            outs, errs = proc.communicate(timeout=15)
            Flash, RAM = re.findall(r'\d+', str(outs))
        except subprocess.TimeoutExpired:
            proc.kill()
            outs, errs = proc.communicate()
            print("stm32tflm error")
            exit()

        return int(Flash), int(RAM)

    def evaluate_model_process(self, k, c) :
        if k > 0 :
            self.model_name = f"k_{k}_c_{c}"
            print(f"\n{self.model_name}\n")
            checkpoint = tf.keras.callbacks.ModelCheckpoint(
                f"{self.path_to_trained_models}/{self.model_name}.h5", monitor='val_accuracy',
                verbose=1, save_best_only=True, save_weights_only=False, mode='auto')
            model, MACC, number_of_cells_limited = self.Model(k, c)
            #One epoch of training must be done before quantization, which is needed to evaluate RAM and Flash occupancy
            model.fit(self.train_ds, epochs=1, validation_data=self.validation_ds, validation_freq=1)
            model.save(f"{self.path_to_trained_models}/{self.model_name}.h5")
            Flash, RAM = self.evaluate_flash_and_peak_RAM_occupancy()
            print(f"\nRAM: {RAM},\t Flash: {Flash},\t MACC: {MACC}\n")
            if MACC <= self.max_MACC and Flash <= self.max_Flash and RAM <= self.max_RAM and not number_of_cells_limited :
                hist = model.fit(self.train_ds, epochs=self.epochs - 1, validation_data=self.validation_ds, validation_freq=1, callbacks=[checkpoint])
                self.quantize_model_uint8()
            return {'k': k,
                    'c': c if not number_of_cells_limited else "Not feasible",
                    'RAM': RAM if RAM <= self.max_RAM else "Outside the upper bound",
                    'Flash': Flash if Flash <= self.max_Flash else "Outside the upper bound",
                    'MACC': MACC if MACC <= self.max_MACC else "Outside the upper bound",
                    'max_val_acc':
                    np.around(np.amax(hist.history['val_accuracy']), decimals=3)
                    if 'hist' in locals() else -3}
        else :
            return{'k': 'unfeasible', 'c': c, 'max_val_acc': -3}

    def explore_num_cells(self, k) :
        previous_architecture = {'k': -1, 'c': -1, 'max_val_acc': -2}
        current_architecture = {'k': -1, 'c': -1, 'max_val_acc': -1}
        c = -1
        k = int(k)

        while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc']) :
            previous_architecture = current_architecture
            c = c + 1
            self.model_counter = self.model_counter + 1
            current_architecture = self.evaluate_model_process(k, c)
            print(f"\n\n\n{current_architecture}\n\n\n")
        return previous_architecture

    def search(self) :
        self.model_counter = 0
        epsilon = 0.005
        k0 = 4

        start = datetime.datetime.now()

        k = k0
        previous_architecture = self.explore_num_cells(k)
        k = 2 * k
        current_architecture = self.explore_num_cells(k)

        if (current_architecture['max_val_acc'] > previous_architecture['max_val_acc']) :
            previous_architecture = current_architecture
            k = 2 * k
            current_architecture = self.explore_num_cells(k)
            while(current_architecture['max_val_acc'] > previous_architecture['max_val_acc'] + epsilon) :
                previous_architecture = current_architecture
                k = 2 * k
                current_architecture = self.explore_num_cells(k)
        else :
            k = k0 / 2
            current_architecture = self.explore_num_cells(k)
            while(current_architecture['max_val_acc'] >= previous_architecture['max_val_acc']) :
                previous_architecture = current_architecture
                k = k / 2
                current_architecture = self.explore_num_cells(k)

        resulting_architecture = previous_architecture

        end = datetime.datetime.now()

        if (resulting_architecture['max_val_acc'] > 0) :
            resulting_architecture_name = f"k_{resulting_architecture['k']}_c_{resulting_architecture['c']}.tflite"
            self.path_to_resulting_architecture = f"{self.save_path}/resulting_architecture_{resulting_architecture_name}"
            os.rename(f"{self.path_to_trained_models}/{resulting_architecture_name}", self.path_to_resulting_architecture)
            os.system(f"rm -rf {self.path_to_trained_models}")
            print(f"\nResulting architecture: {resulting_architecture}\n")
        else :
            print(f"\nNo feasible architecture found\n")
        print(f"Elapsed time (search): {end-start}\n")

        return self.path_to_resulting_architecture

In [ ]:
def test_tflite_model(path_to_resulting_architecture, test_ds) :
    interpreter = tf.lite.Interpreter(path_to_resulting_architecture)
    interpreter.allocate_tensors()

    output = interpreter.get_output_details()[0]  # Model has single output.
    input = interpreter.get_input_details()[0]  # Model has single input.

    correct = 0
    wrong = 0

    for image, label in test_ds :
        # Check if the input type is quantized, then rescale input data to uint8
        if input['dtype'] == tf.uint8:
            input_scale, input_zero_point = input["quantization"]
            image = image / input_scale + input_zero_point
        input_data = tf.dtypes.cast(image, tf.uint8)
        interpreter.set_tensor(input['index'], input_data)
        interpreter.invoke()
        if label.numpy().argmax() == interpreter.get_tensor(output['index']).argmax() :
            correct = correct + 1
        else :
            wrong = wrong + 1
    print(f"\nTflite model test accuracy: {correct/(correct+wrong)}")


load STM's program to evaluate peak RAM occupancy and memory Occupancy of Tflite models. It can be found inside the linux package of X-CUBE-AI at the following path: "path/to/en.x-cube-ai-linux_v8.0.1/stm32ai-linux-8.0.1/linux/stm32tflm" . The package can be downloaded at https://www.st.com/en/embedded-software/x-cube-ai.html#get-software.

In [ ]:
path_to_project = '/content/drive/MyDrive/FutureGenerationComputerSystems2'

In [ ]:
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)

In [ ]:
path_to_stm32_script = path_to_project + '/stm32tflm'

In [ ]:
!cp $path_to_stm32_script /content/

enable its execution on this virtual machine

In [ ]:
!chmod +x stm32tflm

# Load Datasets

and apply a 0.2 test split (80% for training, 20% for testing) to the datasets which do not provide test sets.

In [ ]:
test_split = 0.2

In [ ]:
import os

def split_dataset(path_to_dataset, test_split) :
  directories = [e for e in os.scandir(path_to_dataset) if e.is_dir()]

  for directory in directories :
    train_directory = path_to_dataset + '/train/' + directory.name
    test_directory = path_to_dataset + '/test/' + directory.name
    os.makedirs(train_directory)
    os.makedirs(test_directory)

    files = [e for e in os.scandir(directory) if os.path.isfile(e)]
    treshold = len(files) * (1 - test_split)
    count = 0

    for f in files :
      if count < treshold :
        os.rename(f.path, f"{train_directory}/{f.name}")
      else :
        os.rename(f.path, f"{test_directory}/{f.name}")
      count = count + 1

    os.rmdir(directory.path)

## Visual Wake Words

https://arxiv.org/pdf/1906.05721.pdf

In [ ]:
path_to_dataset = path_to_project + '/datasets/visual_wake_words.zip'

In [ ]:
!unzip -q $path_to_dataset -d datasets

## Humans

In [ ]:
import os

source_directory = 'datasets/visual_wake_words/maxi_train'
humans_dataset_directory = 'datasets/humans'
num_images = 10000
test_split = 0.2

directories = [e for e in os.scandir(source_directory) if e.is_dir()]

for directory in directories :
  new_dir = f"{humans_dataset_directory}/{directory.name}"
  os.makedirs(new_dir)
  files = [e for e in os.scandir(directory) if os.path.isfile(e)]
  for f in files[0:num_images] :
    os.system(f"cp {f.path} {new_dir}")

split_dataset(humans_dataset_directory, test_split)

## Melanoma Cancer

https://www.kaggle.com/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images?resource=download

In [ ]:
path_to_dataset = path_to_project + '/datasets/melanoma_cancer_dataset.zip'

In [ ]:
!unzip -q $path_to_dataset -d datasets

## Flowers-4

https://www.kaggle.com/datasets/l3llff/flowers

In [ ]:
path_to_dataset = path_to_project + '/datasets/flowers.zip'

In [ ]:
!unzip -q $path_to_dataset -d datasets

delete unwanted classes

In [ ]:
!rm -r datasets/flowers/astilbe
!rm -r datasets/flowers/bellflower
!rm -r datasets/flowers/black_eyed_susan
!rm -r datasets/flowers/calendula
!rm -r datasets/flowers/california_poppy
!rm -r datasets/flowers/carnation
!rm -r datasets/flowers/common_daisy
!rm -r datasets/flowers/coreopsis
!rm -r datasets/flowers/daffodil
!rm -r datasets/flowers/rose
!rm -r datasets/flowers/sunflower
!rm -r datasets/flowers/water_lily

create test and training splits

In [ ]:
test_split = 0.2

split_dataset('datasets/flowers', test_split)

## Animals-3

In [ ]:
path_to_dataset = path_to_project + '/datasets/animals.zip'

In [ ]:
!unzip -q $path_to_dataset -d datasets

remove unwanted files

In [ ]:
!rm -r datasets/animals/translate.py

remove unwanted classes

In [ ]:
!rm -r datasets/animals/raw-img/cane
!rm -r datasets/animals/raw-img/ragno
!rm -r datasets/animals/raw-img/elefante
!rm -r datasets/animals/raw-img/gatto
!rm -r datasets/animals/raw-img/scoiattolo
!rm -r datasets/animals/raw-img/pecora
!rm -r datasets/animals/raw-img/mucca

In [ ]:
test_split = 0.2

split_dataset('datasets/animals/raw-img', test_split)

## MNIST

https://github.com/teavanist/MNIST-JPG

In [ ]:
path_to_dataset = path_to_project + '/datasets/MNIST.zip'

In [ ]:
!mkdir datasets

In [ ]:
!unzip -q $path_to_dataset -d datasets/MNIST

# Experiments

## Comparison with Transfer Learning

### Some useful functions

In [ ]:
import tensorflow as tf
import os

def load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape) :
    num_classes = len(next(os.walk(path_to_training_set))[1])

    train_ds = tf.keras.utils.image_dataset_from_directory(
        directory= path_to_training_set,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        batch_size= batch_size,
        image_size=(input_shape[0], input_shape[1]),
        shuffle=True,
        seed=11,
        validation_split=validation_split,
        subset='training'
    )

    validation_ds = tf.keras.utils.image_dataset_from_directory(
        directory= path_to_training_set,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        batch_size= batch_size,
        image_size=(input_shape[0], input_shape[1]),
        shuffle=True,
        seed=11,
        validation_split=validation_split,
        subset='validation'
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory= path_to_test_set,
        labels='inferred',
        label_mode='categorical',
        color_mode='rgb',
        batch_size= batch_size,
        image_size=(input_shape[0], input_shape[1]),
        shuffle=True,
        seed=11
    )

    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
    validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

    return train_ds, validation_ds, test_ds, num_classes

def quantize_model_uint8(train_ds, model_name) :
    def representative_dataset():
        for data in train_ds.rebatch(1).take(150) :
            yield [tf.dtypes.cast(data[0], tf.float32)]

    model = tf.keras.models.load_model(f"{model_name}.h5")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_quant_model = converter.convert()

    with open(f"{model_name}.tflite", 'wb') as f:
        f.write(tflite_quant_model)

    os.remove(f"{model_name}.h5")

### Melanoma Cancer

#### Transfer Learning with Fine Tuning

https://keras.io/guides/transfer_learning/

In [ ]:
import tensorflow as tf
from datetime import datetime

save_path = save_path = path_to_project + '/results/comparison_with_transfer_learning/transfer_learning/melanoma_cancer'
path_to_training_set = './datasets/melanoma_cancer_dataset/train'
path_to_test_set = './datasets/melanoma_cancer_dataset/test'

batch_size = 128
epochs_transfer_learning = 20
epochs_fine_tuning = 10
validation_split = 0.3
input_shape = (224, 224, 3)

train_ds, validation_ds, test_ds, num_classes = load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape)

#model definition
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",  # Load weights pre-trained on ImageNet.
    input_shape=input_shape,
    include_top=False)

base_model.trainable = False

inputs = tf.keras.Input(shape=input_shape)
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
x = tf.keras.layers.Rescaling(1./255)(x)
x = tf.keras.layers.BatchNormalization()(x)
# The base model contains batchnorm layers. We want to keep them in inference mode
# when we unfreeze the base model for fine-tuning, so we make sure that the
# base_model is running in inference mode here.
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

!nvidia-smi

start = datetime.now()

model.fit(train_ds, epochs=epochs_transfer_learning,
          validation_data=validation_ds, verbose=1, validation_freq=1)

base_model.trainable = True
model.summary()

opt = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

checkpoint = tf.keras.callbacks.ModelCheckpoint(save_path + '.h5',
              monitor='val_accuracy', verbose=1, save_best_only=True,
              save_weights_only=False, mode='auto')

model.fit(train_ds, epochs=epochs_fine_tuning, validation_data=validation_ds,
          callbacks=[checkpoint], verbose=1, validation_freq=1)

end = datetime.now()

print('\n' + 'Training time: ' + str(end-start) + '\n')

print('Test Accuracy: \n')

model.evaluate(test_ds)

quantize_model_uint8(train_ds, save_path)

path_to_tflite_model = save_path + '.tflite'

In [ ]:
!./stm32tflm $path_to_tflite_model

#### ColabNAS

let's run ColabNAS with fine-tuned MobileNetV2's parameters as upper bounds

In [ ]:
#from ColabNAS import ColabNAS
import numpy as np
import tensorflow as tf

input_shape = (224,224,3)

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg

path_to_training_set = './datasets/melanoma_cancer_dataset/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

save_path = path_to_project + '/results/comparison_with_transfer_learning/ColabNAS/melanoma_cancer'

#Fine-tuned MobileNetV2's params
peak_RAM_upper_bound = 2583552
Flash_upper_bound = 2713592
MACC_upper_bound = 300000000 #https://arxiv.org/pdf/1801.04381.pdf

!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
print('\n')

path_to_test_set = './datasets/melanoma_cancer_dataset/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=(224,224),
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### Animals-3

#### Transfer Learning with Fine Tuning

https://keras.io/guides/transfer_learning/

In [ ]:
import tensorflow as tf
from datetime import datetime

save_path = save_path = path_to_project + '/results/comparison_with_transfer_learning/transfer_learning/animals'
path_to_training_set = './datasets/animals/raw-img/train'
path_to_test_set = './datasets/animals/raw-img/test'

batch_size = 128
epochs_transfer_learning = 20
epochs_fine_tuning = 10
validation_split = 0.3
input_shape = (224, 224, 3)

train_ds, validation_ds, test_ds, num_classes = load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape)

#model definition
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",  # Load weights pre-trained on ImageNet.
    input_shape=input_shape,
    include_top=False)

base_model.trainable = False

inputs = tf.keras.Input(shape=input_shape)
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
x = tf.keras.layers.Rescaling(1./255)(x)
x = tf.keras.layers.BatchNormalization()(x)
# The base model contains batchnorm layers. We want to keep them in inference mode
# when we unfreeze the base model for fine-tuning, so we make sure that the
# base_model is running in inference mode here.
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

!nvidia-smi

start = datetime.now()

model.fit(train_ds, epochs=epochs_transfer_learning,
          validation_data=validation_ds, verbose=1, validation_freq=1)

base_model.trainable = True
model.summary()

opt = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

checkpoint = tf.keras.callbacks.ModelCheckpoint(save_path + '.h5',
              monitor='val_accuracy', verbose=1, save_best_only=True,
              save_weights_only=False, mode='auto')

model.fit(train_ds, epochs=epochs_fine_tuning, validation_data=validation_ds,
          callbacks=[checkpoint], verbose=1, validation_freq=1)

end = datetime.now()

print('\n' + 'Training time: ' + str(end-start) + '\n')

print('Test Accuracy: \n')

model.evaluate(test_ds)

quantize_model_uint8(train_ds, save_path)

path_to_tflite_model = save_path + '.tflite'

In [ ]:
!./stm32tflm $path_to_tflite_model

#### ColabNAS

let's run ColabNAS with fine-tuned MobileNetV2's parameters as upper bounds

In [ ]:
#from ColabNAS import ColabNAS
import numpy as np
import tensorflow as tf

input_shape = (224,224,3)

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg

path_to_training_set = './datasets/animals/raw-img/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

save_path = path_to_project + '/results/comparison_with_transfer_learning/ColabNAS/animals'

#Fine-tuned MobileNetV2's params
peak_RAM_upper_bound = 2583552
Flash_upper_bound = 2713592
MACC_upper_bound = 300000000 #https://arxiv.org/pdf/1801.04381.pdf

!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
print('\n')

path_to_test_set = './datasets/animals/raw-img/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=(224,224),
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### Flowers-4

#### Transfer Learning with Fine Tuning

https://keras.io/guides/transfer_learning/

In [ ]:
import tensorflow as tf
from datetime import datetime

save_path = save_path = path_to_project + '/results/comparison_with_transfer_learning/transfer_learning/flowers'
path_to_training_set = './datasets/flowers/train'
path_to_test_set = './datasets/flowers/test'

batch_size = 128
epochs_transfer_learning = 20
epochs_fine_tuning = 10
validation_split = 0.3
input_shape = (224, 224, 3)

train_ds, validation_ds, test_ds, num_classes = load_dataset(path_to_training_set, path_to_test_set, batch_size, validation_split, input_shape)

#model definition
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",  # Load weights pre-trained on ImageNet.
    input_shape=input_shape,
    include_top=False)

base_model.trainable = False

inputs = tf.keras.Input(shape=input_shape)
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.2, fill_mode='constant', interpolation='bilinear')(x)
x = tf.keras.layers.Rescaling(1./255)(x)
x = tf.keras.layers.BatchNormalization()(x)
# The base model contains batchnorm layers. We want to keep them in inference mode
# when we unfreeze the base model for fine-tuning, so we make sure that the
# base_model is running in inference mode here.
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

!nvidia-smi

start = datetime.now()

model.fit(train_ds, epochs=epochs_transfer_learning,
          validation_data=validation_ds, verbose=1, validation_freq=1)

base_model.trainable = True
model.summary()

opt = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(optimizer=opt,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

checkpoint = tf.keras.callbacks.ModelCheckpoint(save_path + '.h5',
              monitor='val_accuracy', verbose=1, save_best_only=True,
              save_weights_only=False, mode='auto')

model.fit(train_ds, epochs=epochs_fine_tuning, validation_data=validation_ds,
          callbacks=[checkpoint], verbose=1, validation_freq=1)

end = datetime.now()

print('\n' + 'Training time: ' + str(end-start) + '\n')

print('Test Accuracy: \n')

model.evaluate(test_ds)

quantize_model_uint8(train_ds, save_path)

path_to_tflite_model = save_path + '.tflite'

In [ ]:
!./stm32tflm $path_to_tflite_model

#### ColabNAS

let's run ColabNAS with fine-tuned MobileNetV2's parameters as upper bounds

In [ ]:
#from ColabNAS import ColabNAS
import numpy as np
import tensorflow as tf

input_shape = (224,224,3)

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg

path_to_training_set = './datasets/flowers/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

save_path = path_to_project + '/results/comparison_with_transfer_learning/ColabNAS/flowers'

#Fine-tuned MobileNetV2's params
peak_RAM_upper_bound = 2583552
Flash_upper_bound = 2713592
MACC_upper_bound = 300000000 #https://arxiv.org/pdf/1801.04381.pdf

!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
print('\n')

path_to_test_set = './datasets/flowers/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=(224,224),
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

## Hardware-awareness proof

### Melanoma Cancer

#### STM32L0

STM32L010RBT6

75 CoreMark, 20 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L010RBT6
#75 CoreMark, 20 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 20480
Flash_upper_bound = 131172
MACC_upper_bound = 750000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/melanoma_cancer_dataset/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/melanoma_cancer/STM32L010RBT6'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/melanoma_cancer_dataset/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L1

STM32L151UCY6DTR

93 CoreMark, 32 kiB RAM, 256 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L151UCY6DTR
#93 CoreMark, 32 kiB RAM, 256 kiB Flash
peak_RAM_upper_bound = 32768
Flash_upper_bound = 262144
MACC_upper_bound = 930000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/melanoma_cancer_dataset/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/melanoma_cancer/STM32L151UCY6DTR'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/melanoma_cancer_dataset/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L4

STM32L412KBU3

273 CoreMark, 40 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L412KBU3
#273 CoreMark, 40 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 40960
Flash_upper_bound = 131072
MACC_upper_bound = 2730000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/melanoma_cancer_dataset/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/melanoma_cancer/STM32L412KBU3'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/melanoma_cancer_dataset/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### Visual Wake Words

#### STM32L0

STM32L010RBT6

75 CoreMark, 20 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L010RBT6
#75 CoreMark, 20 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 20480
Flash_upper_bound = 131172
MACC_upper_bound = 750000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/visual_wake_words/maxi_train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/visual_wake_words/STM32L010RBT6'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/visual_wake_words/mini_val'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L1

STM32L151UCY6DTR

93 CoreMark, 32 kiB RAM, 256 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L151UCY6DTR
#93 CoreMark, 32 kiB RAM, 256 kiB Flash
peak_RAM_upper_bound = 32768
Flash_upper_bound = 262144
MACC_upper_bound = 930000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/visual_wake_words/maxi_train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/visual_wake_words/STM32L151UCY6DTR'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/visual_wake_words/mini_val'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L4

STM32L412KBU3

273 CoreMark, 40 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L412KBU3
#273 CoreMark, 40 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 40960
Flash_upper_bound = 131072
MACC_upper_bound = 2730000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/visual_wake_words/maxi_train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/visual_wake_words/STM32L412KBU3'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/visual_wake_words/mini_val'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### Animals-3

#### STM32L0

STM32L010RBT6

75 CoreMark, 20 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L010RBT6
#75 CoreMark, 20 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 20480
Flash_upper_bound = 131172
MACC_upper_bound = 750000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/animals/raw-img/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/animals/STM32L010RBT6'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/animals/raw-img/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L1

STM32L151UCY6DTR

93 CoreMark, 32 kiB RAM, 256 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L151UCY6DTR
#93 CoreMark, 32 kiB RAM, 256 kiB Flash
peak_RAM_upper_bound = 32768
Flash_upper_bound = 262144
MACC_upper_bound = 930000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/animals/raw-img/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/animals/STM32L151UCY6DTR'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/animals/raw-img/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L4

STM32L412KBU3

273 CoreMark, 40 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L412KBU3
#273 CoreMark, 40 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 40960
Flash_upper_bound = 131072
MACC_upper_bound = 2730000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/animals/raw-img/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/animals/STM32L412KBU3'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/animals/raw-img/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### Flowers-4

#### STM32L0

STM32L010RBT6

75 CoreMark, 20 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L010RBT6
#75 CoreMark, 20 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 20480
Flash_upper_bound = 131172
MACC_upper_bound = 750000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/flowers/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/flowers/STM32L010RBT6'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/flowers/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L1

STM32L151UCY6DTR

93 CoreMark, 32 kiB RAM, 256 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L151UCY6DTR
#93 CoreMark, 32 kiB RAM, 256 kiB Flash
peak_RAM_upper_bound = 32768
Flash_upper_bound = 262144
MACC_upper_bound = 930000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/flowers/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/flowers/STM32L151UCY6DTR'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/flowers/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L4

STM32L412KBU3

273 CoreMark, 40 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L412KBU3
#273 CoreMark, 40 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 40960
Flash_upper_bound = 131072
MACC_upper_bound = 2730000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/flowers/train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/flowers/STM32L412KBU3'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/flowers/test'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

### MNIST

#### STM32L0

STM32L010RBT6

75 CoreMark, 20 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L010RBT6
#75 CoreMark, 20 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 20480
Flash_upper_bound = 131172
MACC_upper_bound = 750000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - training'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/MNIST/STM32L010RBT6'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - testing'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L1

STM32L151UCY6DTR

93 CoreMark, 32 kiB RAM, 256 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L151UCY6DTR
#93 CoreMark, 32 kiB RAM, 256 kiB Flash
peak_RAM_upper_bound = 32768
Flash_upper_bound = 262144
MACC_upper_bound = 930000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - training'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/MNIST/STM32L151UCY6DTR'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - testing'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

#### STM32L4

STM32L412KBU3

273 CoreMark, 40 kiB RAM, 128 kiB Flash

In [ ]:
import numpy as np
import tensorflow as tf

input_shape = (50,50,3)

#target: STM32L412KBU3
#273 CoreMark, 40 kiB RAM, 128 kiB Flash
peak_RAM_upper_bound = 40960
Flash_upper_bound = 131072
MACC_upper_bound = 2730000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - training'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/hardware_awareness_proof/MNIST/STM32L412KBU3'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = '/content/datasets/MNIST/MNIST Dataset JPG format/MNIST - JPG - testing'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

## Comparison with state-of-the-art hardware-aware NAS techniques on the Visual Wake Words dataset

In [ ]:
!pip install --quiet wget

In [ ]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
!cat /proc/cpuinfo

In [ ]:
!nvidia-smi

### MCUNET

model name: "mcunet-vww0" aka "mcunet-10fps_vww.tflite"

int8 accuracy: 87.3 (https://github.com/mit-han-lab/mcunet)

In [ ]:
import wget

URL = 'https://hanlab.mit.edu/projects/tinyml/mcunet/release/mcunet-10fps_vww.tflite'

response = wget.download(URL, 'mcunet-10fps_vww.tflite')

In [ ]:
path_to_tflite_model = '/content/mcunet-10fps_vww.tflite'

In [ ]:
!./stm32tflm $path_to_tflite_model

In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(path_to_tflite_model)
interpreter.allocate_tensors()

%timeit interpreter.invoke()

### Micronets

model name: "MicroNet VWW-2 INT8" aka "vww2_50_50_INT8.tflite"

int8 accuracy: 0.768 (https://github.com/ARM-software/ML-zoo)

In [ ]:
import wget

URL = 'https://github.com/ARM-software/ML-zoo/raw/master/models/visual_wake_words/micronet_vww2/tflite_int8/vww2_50_50_INT8.tflite'

response = wget.download(URL, 'vww2_50_50_INT8.tflite')

In [ ]:
path_to_tflite_model = '/content/vww2_50_50_INT8.tflite'

In [ ]:
!./stm32tflm $path_to_tflite_model

In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(path_to_tflite_model)
interpreter.allocate_tensors()

%timeit interpreter.invoke()

### ColabNAS

In [ ]:
import numpy as np
import tensorflow as tf

#target: STM32F412 (the same of mcunet-vww0)
#608 CoreMark, 256 kiB RAM, 1024 kiB Flash
input_shape = (50,50,3)
peak_RAM_upper_bound = 131072
Flash_upper_bound = 524288
MACC_upper_bound = 6080000 #CoreMark * 1e4

#Each dataset must comply with the following structure
#main_directory/
#...class_a/
#......a_image_1.jpg
#......a_image_2.jpg
#...class_b/
#......b_image_1.jpg
#......b_image_2.jpg
path_to_training_set = './datasets/visual_wake_words/maxi_train'
val_split = 0.3

#whether or not to cache datasets in memory
#if the dataset cannot fit in the main memory, the application will crash
cache = True

#where to save results
save_path = path_to_project + '/results/comparison_with_the_state_of_the_art'

#to show the GPU used
!nvidia-smi

colabNAS = ColabNAS(peak_RAM_upper_bound, Flash_upper_bound, MACC_upper_bound, path_to_training_set, val_split, cache, input_shape, save_path=save_path)

#search
path_to_tflite_model = colabNAS.search()

#test
path_to_test_set = './datasets/visual_wake_words/mini_val'

test_ds = tf.keras.utils.image_dataset_from_directory(
    directory= path_to_test_set,
    labels='inferred',
    label_mode='categorical',
    color_mode='rgb',
    batch_size=1,
    image_size=input_shape[0:2],
    shuffle=True
)

test_tflite_model(path_to_tflite_model, test_ds)

In [ ]:
!./stm32tflm $path_to_tflite_model

In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(path_to_tflite_model)
interpreter.allocate_tensors()

%timeit interpreter.invoke()